# 05 – Conclude & Evaluate

**Phase:** C – Conclude (QUA³CK)  
**Projekt:** WealthScope AI  

---

## Zielsetzung dieser Phase

Die C-Phase bewertet alle vorangegangenen Phasen und zieht ein  
wissenschaftlich fundiertes Fazit über Stärken, Grenzen und Ausblick.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, confusion_matrix)
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": "white"})

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
df = pd.read_parquet(DATA_DIR / "wealthscope_features.parquet")

FEATURES = [c for c in ["daily_return","return_5d","return_20d",
             "ma_20_distance","ma_50_distance","ma_200_distance",
             "volatility_20d","drawdown"] if c in df.columns]
TARGET = "target_20d"

model_df = df[FEATURES + [TARGET]].dropna(subset=[TARGET])
X = model_df[FEATURES]
y = model_df[TARGET].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25,
                                                      random_state=42, stratify=y)

def pipe(model):
    return Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("scl", StandardScaler()), ("mod", model)])

models = {
    "Majority Baseline": pipe(DummyClassifier(strategy="most_frequent")),
    "Logistische Regression": pipe(LogisticRegression(max_iter=1000, class_weight="balanced")),
    "Random Forest": pipe(RandomForestClassifier(n_estimators=200, max_depth=8,
                                                   class_weight="balanced", random_state=42, n_jobs=-1)),
}
results = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    proba = m.predict_proba(X_test)[:,1]
    cv = cross_val_score(m, X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1)
    results[name] = {
        "accuracy": accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred, average="weighted", zero_division=0),
        "auc": roc_auc_score(y_test, proba),
        "cv_auc_mean": cv.mean(),
        "cv_auc_std": cv.std(),
    }

summary = pd.DataFrame(results).T.round(4)
print("=" * 65)
print("  FINALE MODELLEVALUATION – WealthScope AI")
print("=" * 65)
print(summary.to_string())
print("=" * 65)


## Gesamtbewertung des Projekts

### Ergebnisse der QUA³CK-Phasen

| Phase | Status | Kernergebnis |
|---|---|---|
| **Q – Question** | ✅ | Forschungsfrage klar definiert, Hypothesen aufgestellt |
| **U – Understanding** | ✅ | EDA, fehlende Werte, Imputation, Scaling durchgeführt |
| **A – Analytics** | ✅ | 8 technische Features engineered |
| **A – Algorithm** | ✅ | 3 Modelle verglichen (Baseline, LogReg, RF) |
| **A – Adaption** | ✅ | Cross-Validation, Hyperparameter angepasst |
| **C – Conclude** | ✅ | Dieses Notebook |
| **K – Knowledge** | ✅ | Streamlit-App + NewsAPI-Integration |

### Modell-Fazit

Das Random-Forest-Modell erreicht ~53 % Accuracy und AUC ~0.53.  
Dies liegt leicht über Zufall (50 %) und ist für historische technische Daten **scientifically expected**.

**Warum nicht besser?**
1. **Efficient Market Hypothesis** (Fama 1970): Öffentlich verfügbare historische Kursdaten  
   sind bereits im Marktpreis eingepreist.
2. **Rauschen überwiegt**: Finanzmärkte haben ein sehr niedriges Signal/Rausch-Verhältnis.
3. **Fehlende exogene Faktoren**: Makroökonomie, Sentiment, Nachrichten nicht einbezogen.

Das Modell wird in der App **nicht als Prognose** kommuniziert, sondern als ein Baustein  
im regelbasierten Confidence-Score – eine methodisch korrekte Einbettung.

### Stärken des Projekts

- ✅ Vollständige QUA³CK-Dokumentation über 8 Notebooks
- ✅ Saubere Data-Leakage-freie Pipeline
- ✅ Interaktive Streamlit-App mit echten Daten
- ✅ Wissenschaftliche Quellen und Disclaimer
- ✅ Transparente Kommunikation der Modellgrenzen

### Schwächen & Ausblick

| Limitation | Mögliche Verbesserung |
|---|---|
| Nur technische Features | + Sentiment (NewsAPI), + Makrodaten |
| Random Forest | + XGBoost, LightGBM, LSTM |
| Statisches Modell | + Online Learning, regelmäßiges Retraining |
| Keine Live-Daten | + Broker-API Integration (Ausblick) |
| Accuracy ~53 % | Erwarteter Wert – kein Bug |

### Abschluss-Statement

> *WealthScope AI erfüllt den QUA³CK-Prozess vollständig: Aus einer präzisen Forschungsfrage  
> über systematisches Datenverständnis und transparente Modellierung entsteht eine  
> interaktive Anwendung, die Ergebnisse kommuniziert und Wissen über datengetriebene  
> Finanzanalyse nachvollziehbar vermittelt.*

---

### Quellen

- Fama (1970): Efficient Capital Markets. Journal of Finance.
- Stock et al. (2021): QUA³CK Process Model. KIT. https://publikationen.bibliothek.kit.edu/1000129631
- Scikit-learn (2025): https://scikit-learn.org
- WealthScope AI Projektdokumentation: docs/qua3ck_process.md
